In [1]:
import nltk
nltk.download("punkt", quiet= True) # cümle/kelime tokonizer verisi (word_tokenize için)
nltk.download("stopwords", quiet= True) # durak kelime listeleri (the, and , is, ...) - filtrelemek için
nltk.download("wordnet", quiet= True) # lemmatization sözlüğü (WordNetLemmatizer için kök bilgisi)
nltk.download("averaged_perceptron_tagger", quiet= True) # POS tagger verisi (kelime türü: isim/fiil...)

True

In [2]:
# Stopwords çıkarma
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

stop_en = set(stopwords.words("english"))
text = "this is a simple sentence for testing nlp pipeline"

tokens = word_tokenize(text)
# stopword OLMAYAN kelimeleri tek tek topla
clean = []
for t in tokens:
    if t.lower() not in stop_en:
        clean.append(t)
print(f"\nHam: {tokens}")
print(f"\nNo SW: {clean}")


Ham: ['this', 'is', 'a', 'simple', 'sentence', 'for', 'testing', 'nlp', 'pipeline']

No SW: ['simple', 'sentence', 'testing', 'nlp', 'pipeline']


In [3]:
turkish = "Bu bir örnek cümledir ve Türkçe işleniyor"
tokens = word_tokenize(turkish, language= "turkish")
stop_tr = set(stopwords.words("turkish"))
clean = []
for t in tokens:
    if t.lower() not in stop_tr:
        clean.append(t)

print(f"\nTürkçe ham: {tokens}")
print(f"Türkçe no SW: {clean}")


Türkçe ham: ['Bu', 'bir', 'örnek', 'cümledir', 've', 'Türkçe', 'işleniyor']
Türkçe no SW: ['bir', 'örnek', 'cümledir', 'Türkçe', 'işleniyor']


In [6]:
# TAM PIPELINE FONKSIYONU
def nlp_pipeline(text, lang = "english"):
    # 1. Lowercase
    text = text.lower()
    # 2. Tokenize
    tokens = word_tokenize(text, language= lang)
    # 3. Stopword + noktalama
    sw = set(stopwords.words(lang))
    temiz = []
    for t in tokens:
        if t.isalpha() and t not in sw:
            temiz.append(t)
    tokens = temiz
    # 4. Lemmatize
    if lang == "english":
        lemma = WordNetLemmatizer()
        kokler = []
        for t in tokens:
            kokler.append(lemma.lemmatize(t, pos = "v"))
        tokens = kokler
    return tokens


In [7]:
print(nlp_pipeline("The quick brown foxes were running quickly over the lazy dogs"))

['quick', 'brown', 'fox', 'run', 'quickly', 'lazy', 'dog']


# Türkçe Lemmatization: WordNet neden yetmez -> Zeyrek

İngilizce için WordNet lemmatizer işe yaradı. Ama Türkçe eklemeli bir dil: kitap+lar+ımız+dan. WordNet İngilizce bir sözlük olduğu için Türkçe kelimeyi olduğu gibi verir. - kökü bulamaz. Türkçe için Zeyrek kullanırız

In [8]:
lemma = WordNetLemmatizer()

print("WordNet İngilzice sözlük - Türkçe kökü bulamaz:")
print(f" kitaplarımızdan -> {lemma.lemmatize('kitaplarımızdan')}") # değişmedi
print(f" koşuyorlar -> {lemma.lemmatize('koşuyprlar')}") # değişmedi
print(f" güzeldi -> {lemma.lemmatize('güzeldi')}") # değişmedi

WordNet İngilzice sözlük - Türkçe kökü bulamaz:
 kitaplarımızdan -> kitaplarımızdan
 koşuyorlar -> koşuyprlar
 güzeldi -> güzeldi


In [11]:
# Zeyrek - Türkçe morfolojik lemmatizer
from zeyrek import MorphAnalyzer

analiz = MorphAnalyzer()

def zeyrek_kok(kelime):
    sonuc = analiz.lemmatize(kelime)
    if sonuc and sonuc[0][1]:
        return sonuc[0][1][0].lower() # ilk analiz, ilk kök, küçük harf
    return kelime

print("Zeyrek Türkçe kökleri buluyor:")
print(f" kitaplarımızdan -> {zeyrek_kok('kitaplarımızdan')}") # değişmedi
print(f" koşuyorlar -> {zeyrek_kok('koşuyorlar')}") # değişmedi
print(f" güzeldi -> {zeyrek_kok('güzeldi')}") # değişmedi

APPENDING RESULT: <(Kitap_Noun_Prop)(-)(kitap:nounProper_S + lar:a3pl_S + ımız:p1pl_S + dan:abl_ST)>
APPENDING RESULT: <(kitap_Noun)(-)(kitap:noun_S + lar:a3pl_S + ımız:p1pl_S + dan:abl_ST)>
APPENDING RESULT: <(koşmak_Verb)(-)(koş:verbRoot_S + uyor:vProgYor_S + lar:vA3pl_ST)>
APPENDING RESULT: <(güzel_Adj)(-)(güzel:adjectiveRoot_ST + adjZeroDeriv_S + nVerb_S + di:nPast_S + nA3sg_ST)>
APPENDING RESULT: <(güzel_Noun)(-)(güzel:noun_S + a3sg_S + pnon_S + nom_ST + nounZeroDeriv_S + nVerb_S + di:nPast_S + nA3sg_ST)>


Zeyrek Türkçe kökleri buluyor:
 kitaplarımızdan -> kitap
 koşuyorlar -> koşmak
 güzeldi -> güzel


In [12]:
kelimeler = ["kitaplarımızdan","evlerinde","koşuyorlar","güzeldi","arabası"]

print(f"{'Kelime':<18} {'WordNet (EN)':<20}{'Zeyrek (TR)'}")
print("-" * 52)
for k in kelimeler:
    print(f"{k:<18}{lemma.lemmatize(k):<20} {zeyrek_kok(k)}")

    # Sonuç: WordNet Türkçe'de kelimeyi değiştirmez; Zeyrek gerçek kökü verir.

APPENDING RESULT: <(Kitap_Noun_Prop)(-)(kitap:nounProper_S + lar:a3pl_S + ımız:p1pl_S + dan:abl_ST)>
APPENDING RESULT: <(kitap_Noun)(-)(kitap:noun_S + lar:a3pl_S + ımız:p1pl_S + dan:abl_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + a3sg_S + leri:p3pl_S + nde:loc_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + ler:a3pl_S + in:p2sg_S + de:loc_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + ler:a3pl_S + i:p3sg_S + nde:loc_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + ler:a3pl_S + i:p3pl_S + nde:loc_ST)>
APPENDING RESULT: <(koşmak_Verb)(-)(koş:verbRoot_S + uyor:vProgYor_S + lar:vA3pl_ST)>
APPENDING RESULT: <(güzel_Adj)(-)(güzel:adjectiveRoot_ST + adjZeroDeriv_S + nVerb_S + di:nPast_S + nA3sg_ST)>
APPENDING RESULT: <(güzel_Noun)(-)(güzel:noun_S + a3sg_S + pnon_S + nom_ST + nounZeroDeriv_S + nVerb_S + di:nPast_S + nA3sg_ST)>
APPENDING RESULT: <(araba_Noun)(-)(araba:noun_S + a3sg_S + sı:p3sg_S + nom_ST)>
APPENDING RESULT: <(araba_Adj)(-)(araba:adjectiveRoot_ST + adjZeroDeriv_S + noun_

Kelime             WordNet (EN)        Zeyrek (TR)
----------------------------------------------------
kitaplarımızdan   kitaplarımızdan      kitap
evlerinde         evlerinde            ev
koşuyorlar        koşuyorlar           koşmak
güzeldi           güzeldi              güzel
arabası           arabası              araba
